# 04. Побудова ознак (Feature Engineering)

На даному етапі виконується формування та аналіз додаткових ознак,
які можуть покращити якість моделей машинного навчання.

Основні завдання:

- завантаження підготовлених даних;
- аналіз існуючих ознак;
- створення похідних ознак;
- оцінка їхньої інформативності;
- підготовка фінального набору ознак для моделювання.

In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import sys
import importlib
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


In [3]:
# ============================================================
# LOAD PROJECT
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/FitnessML_Master"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

importlib.invalidate_caches()

print("✓ Project connected")
print(PROJECT_DIR)

✓ Project connected
/content/drive/MyDrive/FitnessML_Master


In [4]:
# ============================================================
# IMPORT PROJECT MODULES
# ============================================================

import config
import utils

config = importlib.reload(config)
utils = importlib.reload(utils)

utils.section("Project modules")

print("✓ config.py loaded")
print("✓ utils.py loaded")


PROJECT MODULES
✓ config.py loaded
✓ utils.py loaded


In [5]:
# ============================================================
# LOAD PREPROCESSED DATA
# ============================================================

utils.section("Preprocessed dataset")

X_train = pd.read_csv(
    config.TABLES_DIR / "03_X_train.csv"
)

X_valid = pd.read_csv(
    config.TABLES_DIR / "03_X_valid.csv"
)

X_test = pd.read_csv(
    config.TABLES_DIR / "03_X_test.csv"
)

y_train = pd.read_csv(
    config.TABLES_DIR / "03_y_train.csv"
)

y_valid = pd.read_csv(
    config.TABLES_DIR / "03_y_valid.csv"
)

y_test = pd.read_csv(
    config.TABLES_DIR / "03_y_test.csv"
)

print(f"Train      : {X_train.shape}")
print(f"Validation : {X_valid.shape}")
print(f"Test       : {X_test.shape}")


PREPROCESSED DATASET
Train      : (292000, 10)
Validation : (36500, 10)
Test       : (36500, 10)


# Аналіз існуючих ознак

Перед створенням нових ознак необхідно оцінити вже наявні характеристики
підготовленого набору даних.

На цьому етапі визначається кількість ознак, їхні типи та базові статистичні
характеристики. Це дозволяє уникнути створення дублюючих або некоректних ознак.

In [ ]:
# ============================================================
# EXISTING FEATURES
# ============================================================

utils.section("Existing features")

print(f"Кількість ознак: {X_train.shape[1]}")

print("\nНазви ознак:")
for i, column in enumerate(X_train.columns, start=1):
    print(f"{i:02d}. {column}")

print("\nТипи даних:")
display(
    X_train.dtypes.to_frame(name="Тип даних")
)

print("\nСтатистичні характеристики:")
display(X_train.describe().T)


EXISTING FEATURES
Кількість ознак: 10

Назви ознак:
01. age
02. steps
03. heart_rate_avg
04. sleep_hours
05. exercise_minutes
06. stress_level
07. weight_kg
08. bmi
09. gender_F
10. gender_M

Типи даних:


,Тип даних
age,float64
steps,float64
heart_rate_avg,float64
sleep_hours,float64
exercise_minutes,float64
stress_level,float64
weight_kg,float64
bmi,float64
gender_F,float64
gender_M,float64



Статистичні характеристики:


,count,mean,std,min,25%,50%,75%,max
age,292000.0,-2.011542e-16,1.000002,-1.730000,-0.887034,0.012129,0.855095,1.698060
steps,292000.0,-8.754033e-17,1.000002,-1.919668,-0.864694,-0.001219,0.863639,1.956637
heart_rate_avg,292000.0,2.080528e-16,1.000002,-4.550623,-0.674317,0.002057,0.673867,4.298680
sleep_hours,292000.0,-1.042794e-15,1.000002,-2.676811,-0.677270,-0.001667,0.676523,3.341725
exercise_minutes,292000.0,1.934526e-17,1.000002,-0.995582,-0.711002,-0.306315,0.384520,15.153977
stress_level,292000.0,-1.145628e-16,1.000002,-1.566081,-0.869975,-0.173869,0.870290,1.566396
weight_kg,292000.0,-9.659001e-16,1.000002,-4.296522,-0.674619,-0.001279,0.675121,4.582572
bmi,292000.0,2.193071e-15,1.000002,-4.399946,-0.674346,-0.000887,0.673273,4.600313
gender_F,292000.0,5.107808e-01,0.499885,0.000000,0.000000,1.000000,1.000000,1.000000
gender_M,292000.0,4.892192e-01,0.499885,0.000000,0.000000,0.000000,1.000000,1.000000


# Аналіз зв'язку ознак із цільовою змінною

На цьому етапі оцінюється зв'язок між наявними ознаками та цільовою
змінною `calories_burned`.

Аналіз дозволяє визначити ознаки, які потенційно мають найбільшу
інформативність для подальшого моделювання, а також сформувати основу
для створення похідних ознак.

In [ ]:
# ============================================================
# FEATURE-TARGET RELATIONSHIP
# ============================================================

utils.section("Feature-target relationship")

target_train = y_train["calories_burned"]

correlations = (
    X_train
    .corrwith(target_train)
    .sort_values(ascending=False)
)

correlation_table = correlations.to_frame(
    name="Кореляція з цільовою змінною"
)

display(correlation_table)


FEATURE-TARGET RELATIONSHIP


,Кореляція з цільовою змінною
heart_rate_avg,0.002918
steps,0.001807
gender_M,0.001784
exercise_minutes,0.001441
weight_kg,0.001329
age,0.000175
bmi,-0.000202
sleep_hours,-0.001704
gender_F,-0.001784
stress_level,-0.003500


# Аналіз нелінійної залежності

Кореляція Пірсона дозволяє оцінити переважно лінійний зв'язок між ознакою
та цільовою змінною.

Оскільки отримані кореляції є дуже низькими, додатково оцінюється
нелінійна залежність за допомогою взаємної інформації (Mutual Information).

Це дозволяє виявити потенційні залежності, які не відображаються
звичайною кореляцією.

In [ ]:
# ============================================================
# NON-LINEAR FEATURE RELATIONSHIP
# ============================================================

from sklearn.feature_selection import mutual_info_regression

utils.section("Non-linear feature relationship")

target_train = y_train["calories_burned"]

mutual_info = mutual_info_regression(
    X_train,
    target_train,
    random_state=config.RANDOM_STATE
)

mutual_info_table = pd.DataFrame({
    "Ознака": X_train.columns,
    "Взаємна інформація": mutual_info
}).sort_values(
    "Взаємна інформація",
    ascending=False
)

display(mutual_info_table)


NON-LINEAR FEATURE RELATIONSHIP


,Ознака,Взаємна інформація
1,steps,0.003191
7,bmi,0.001125
5,stress_level,0.000990
0,age,0.000434
4,exercise_minutes,0.000231
2,heart_rate_avg,0.000155
3,sleep_hours,0.000000
6,weight_kg,0.000000
8,gender_F,0.000000
9,gender_M,0.000000


# Візуалізація інформативності ознак

Результати аналізу взаємної інформації візуалізуються для порівняння
потенційної інформативності ознак.

In [ ]:
# ============================================================
# MUTUAL INFORMATION PLOT
# ============================================================

utils.section("Mutual information")

fig = plt.figure(figsize=(10, 6))

plt.barh(
    mutual_info_table["Ознака"],
    mutual_info_table["Взаємна інформація"]
)

plt.xlabel("Взаємна інформація")
plt.ylabel("Ознака")
plt.title("Нелінійна інформативність ознак")

plt.gca().invert_yaxis()

plt.tight_layout()

utils.save_figure(
    fig,
    "04_mutual_information.png"
)


MUTUAL INFORMATION
✓ Figure saved -> /content/drive/MyDrive/FitnessML_Master/figures/04_mutual_information.png


# Створення похідних ознак

На основі попереднього аналізу створюються додаткові ознаки,
які описують взаємодію між фізичною активністю, пульсом,
тривалістю вправ та іншими характеристиками користувача.

Додатково використовуються квадрати окремих числових ознак
для моделювання потенційних нелінійних залежностей.

In [ ]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

utils.section("Feature engineering")

def create_features(df):
    df = df.copy()

    # --------------------------------------------------------
    # Interaction features
    # --------------------------------------------------------

    df["steps_x_exercise"] = (
        df["steps"] * df["exercise_minutes"]
    )

    df["heart_rate_x_exercise"] = (
        df["heart_rate_avg"] * df["exercise_minutes"]
    )

    df["steps_x_heart_rate"] = (
        df["steps"] * df["heart_rate_avg"]
    )

    df["bmi_x_exercise"] = (
        df["bmi"] * df["exercise_minutes"]
    )

    # --------------------------------------------------------
    # Non-linear features
    # --------------------------------------------------------

    df["steps_squared"] = (
        df["steps"] ** 2
    )

    df["exercise_minutes_squared"] = (
        df["exercise_minutes"] ** 2
    )

    df["heart_rate_squared"] = (
        df["heart_rate_avg"] ** 2
    )

    df["stress_level_squared"] = (
        df["stress_level"] ** 2
    )

    return df


X_train_engineered = create_features(X_train)
X_valid_engineered = create_features(X_valid)
X_test_engineered = create_features(X_test)

print(
    f"Original features : {X_train.shape[1]}"
)

print(
    f"Engineered features: {X_train_engineered.shape[1]}"
)


FEATURE ENGINEERING
Original features : 10
Engineered features: 18


In [ ]:
# ============================================================
# ENGINEERED DATA VALIDATION
# ============================================================

utils.section("Engineered data validation")

assert X_train_engineered.shape[0] == X_train.shape[0]
assert X_valid_engineered.shape[0] == X_valid.shape[0]
assert X_test_engineered.shape[0] == X_test.shape[0]

assert list(X_train_engineered.columns) == list(
    X_valid_engineered.columns
)

assert list(X_train_engineered.columns) == list(
    X_test_engineered.columns
)

assert "calories_burned" not in X_train_engineered.columns

print("✓ Row counts preserved")
print("✓ Feature columns are consistent")
print("✓ Target leakage not detected")

print(
    f"\nFeatures: {X_train_engineered.shape[1]}"
)


ENGINEERED DATA VALIDATION
✓ Row counts preserved
✓ Feature columns are consistent
✓ Target leakage not detected

Features: 18


In [ ]:
# ============================================================
# NEW FEATURES
# ============================================================

utils.section("New features")

original_features = set(X_train.columns)

new_features = [
    column
    for column in X_train_engineered.columns
    if column not in original_features
]

for feature in new_features:
    print(f" - {feature}")


NEW FEATURES
 - steps_x_exercise
 - heart_rate_x_exercise
 - steps_x_heart_rate
 - bmi_x_exercise
 - steps_squared
 - exercise_minutes_squared
 - heart_rate_squared
 - stress_level_squared


In [ ]:
# ============================================================
# SAVE ENGINEERED DATASETS
# ============================================================

utils.section("Save engineered datasets")

X_train_engineered.to_csv(
    config.TABLES_DIR / "04_X_train_engineered.csv",
    index=False
)

X_valid_engineered.to_csv(
    config.TABLES_DIR / "04_X_valid_engineered.csv",
    index=False
)

X_test_engineered.to_csv(
    config.TABLES_DIR / "04_X_test_engineered.csv",
    index=False
)

print("✓ Training features saved")
print("✓ Validation features saved")
print("✓ Test features saved")


SAVE ENGINEERED DATASETS
✓ Training features saved
✓ Validation features saved
✓ Test features saved


In [ ]:
# ============================================================
# NOTEBOOK COMPLETED
# ============================================================

utils.section("Feature engineering completed")


FEATURE ENGINEERING COMPLETED
